# 01 - Create the Kubernetes cluster

Read this notebook and paste each Bash block into an Ubuntu terminal, in order. Keep the same terminal for the commands in this notebook. These are shell instructions; do not run them in a Python kernel.

## Open the project

```bash
export WORKSPACE_HOME="${WORKSPACE_HOME:-$HOME/workspace}"
export PATH="$HOME/.local/bin:$PATH"
cd "$WORKSPACE_HOME"
source versions.env
```



## Start the Minikube profile

```bash
minikube start --profile=bigdata --driver=docker --container-runtime=containerd \
  --kubernetes-version="$LAB_K8S_VERSION" --cpus="$LAB_CPUS" --memory="$LAB_MEMORY_MB"
```

The profile, context and namespace are named `bigdata`. Host Docker contains the node; containerd inside the node contains application images.

## Select the context

```bash
kubectl config use-context bigdata
```



## Wait for the node

```bash
kubectl wait --for=condition=Ready node --all --timeout=180s
```



## Wait for DNS and storage

```bash
kubectl -n kube-system rollout status deployment/coredns --timeout=180s
kubectl -n kube-system wait --for=condition=Ready pod/storage-provisioner --timeout=180s
```

Proceed after DNS and the local storage provisioner are ready.

## Inspect the cluster

```bash
minikube status -p bigdata
kubectl get nodes -o wide
kubectl get pods -n kube-system
kubectl get storageclass
kubectl auth can-i "*" "*" --all-namespaces
```

Expect Ready, healthy system pods, default `standard` storage and `yes` for the permission check.

## Create the namespace

```bash
kubectl apply -f platform/namespace.yaml
```



## Set the default namespace

```bash
kubectl config set-context bigdata --namespace=bigdata
```



## Create the shared configuration

```bash
kubectl apply -f platform/lab-secret.yaml
```

The checked-in lab credentials are `team` / `team1234`; Polaris uses `root` / `s3cr3t`. These are public examples for this local setup.

Start monitoring in notebook 02 before deploying S3, PostgreSQL, Polaris, then MySQL. Polaris depends on PostgreSQL for catalog metadata and S3 for object storage. MySQL is independent and is placed last for a consistent sequence.

Persistent volume names are allocated dynamically; PVC names remain stable. Use `kubectl get pvc,pv` to see the actual mapping. With the Docker driver the Minikube node uses Docker volume `bigdata`; the Kubernetes PVC data lives inside that node volume. Do not edit Docker volumes directly.

## Kubernetes web UI in the Ubuntu desktop

Headlamp is installed by the Minikube addon; no application Deployment YAML is maintained here. The small platform/headlamp-access.yaml defines a repeatable login identity with cluster viewing and Pod management in namespace bigdata. See UI_ACCESS.md for the host exposure script. Use the browser inside the Guacamole Ubuntu desktop. It can access localhost on that Ubuntu machine; no laptop tunnel is needed.

## Enable resource metrics

```bash
minikube -p bigdata addons enable metrics-server
kubectl -n kube-system rollout status deployment/metrics-server --timeout=300s
```



## Enable Headlamp

```bash
minikube -p bigdata addons enable headlamp
kubectl -n headlamp rollout status deployment/headlamp --timeout=300s
```



## Apply UI access permissions

```bash
kubectl apply -f platform/headlamp-access.yaml
```



## Check UI permissions

```bash
kubectl auth can-i list pods -n bigdata --as=system:serviceaccount:headlamp:workspace-ui
kubectl auth can-i get nodes --as=system:serviceaccount:headlamp:workspace-ui
```

Expect yes. This UI identity has view permissions; configuration changes use your normal kubectl account.

## Generate your login token

```bash
kubectl -n headlamp create token workspace-ui --duration=8h
```

Copy the displayed token into Headlamp. Do not save it in notebooks or Git. Generate a fresh token if the API server expires it sooner.

## Open Headlamp in the Ubuntu browser

```bash
minikube -p bigdata service headlamp -n headlamp --url
```

Paste the printed URL into the browser inside the Guacamole Ubuntu desktop on this same host. Log in with your token and select namespace bigdata. On Ubuntu's Docker driver, the NodePort URL is directly reachable from the host; no laptop tunnel or persistent forwarding command is needed. Browse workloads, Services, storage, logs and events. The workspace identity cannot read Secrets directly. Pod management and terminal access are enabled in namespace bigdata.


## Check resource usage

```bash
for attempt in $(seq 1 60); do
  if kubectl top nodes; then break; fi
  sleep 2
done
kubectl top nodes
kubectl top pods -n kube-system
```

Metrics-server supplies current CPU/memory to Headlamp. Prometheus in notebook 02 stores historical metrics. Wait a minute after enabling metrics-server before this check.

## Optional loopback access

If you prefer a stable localhost URL on the Ubuntu desktop, run this in a second terminal:

```bash
kubectl -n headlamp port-forward --address=127.0.0.1 service/headlamp 4466:80
```

Keep it open and browse http://127.0.0.1:4466. This is optional; the NodePort URL above was tested on the Ubuntu host.